In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
import nqetools as nqe
# This follows:
# https://github.com/i-pi/piqm2023-tutorial/blob/main/05-RPI/tutorial-4.ipynb

In [ ]:
# Change the working directory to the location of this script
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "examples", "ch4hcbe")))

# Paths
directory_opti = 'opti'
directory_phonon_react = 'phonon_react'
directory_ts = 'ts'
directory_phonon_ts = 'phonon_ts'
directory_instanton = 'instanton'

directory_opti_d = 'opti_d'
directory_phonon_react_d = 'phonon_react_d'
directory_ts_d = 'ts_d'
directory_phonon_ts_d = 'phonon_ts_d'

# Driver
driver_code = 'cbe'

# Values
temperature = 300.0
n_beads = 10
tol_energy = 5.0e-4
tol_force = 5.0e-4
tol_position = 1.0e-4
total_steps = 1000
optimizer = "cg"


In [ ]:
atoms = nqe.read_ipi_xyz("react.xyz")[-1]
n_atoms = len(atoms)

atoms_ts = nqe.read_ipi_xyz("ts.xyz")[-1]

In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position,)
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
nqe.run_phonons(directory_phonon_react,
                atoms_opti,
                driver=driver_code,
                )

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_react, 'phonon.phonons.eigval'))
fr = nqe.freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
nqe.run_instanton_post_process(directory_phonon_react,
                               process_type='reactant',
                               temperature=temperature,
                               filter_list=n_atoms - 1)

In [ ]:
data = nqe.parse_react_thermo_data(directory_phonon_react)
print(data)

In [ ]:
output = nqe.run_ts(directory_ts,
                    atoms_ts,
                    driver=driver_code,
                    tol_energy=tol_energy,
                    tol_force=tol_force,
                    tol_position=tol_position,)
atoms_ts, output_data_ts, output_desc_ts = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_ts, save=False)

In [ ]:
rate = nqe.calc_forward_rate(n_atoms, directory_ts, directory_phonon_react, temperature, 0.0)
print(rate)

In [ ]:
nqe.run_phonons(directory_phonon_ts,
                atoms_ts,
                driver=driver_code)

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_ts, 'phonon.phonons.eigval'))
fr = nqe.freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
# Run the instanton
nqe.run_instanton(directory_instanton,
                  atoms_ts,
                  directory_ts,
                  driver=driver_code,
                  n_beads=n_beads,
                  temperature=temperature,
                  tol_energy=tol_energy,
                  tol_force=tol_force,
                  tol_position=tol_position,)

In [ ]:
nqe.calc_kappa_full(directory_ts,
                    directory_instanton,
                    temperature,
                    n_beads)

In [ ]:
# Converge over temperature
list_temperature = [100, 150, 200, 250, 300, 350, 400]
list_temperature = [200, 250, 300, 350, 400]
list_kappa = []
n_beads = 40

for temperature in list_temperature:
    print(f"Running with {temperature} K")
    nqe.run_instanton(directory_instanton,
                      atoms_ts,
                      directory_ts,
                      driver=driver_code,
                      n_beads=n_beads,
                      temperature=temperature,
                      tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position,)
    kappa = nqe.calc_kappa_full(directory_ts,
                                directory_instanton,
                                temperature,
                                n_beads=n_beads,
                                react_energy=0.0)
    list_kappa.append(kappa + 1)

In [ ]:
print(list_kappa)
plt.plot(list_temperature, list_kappa, 'o-')
plt.xlabel('Temperature (K)')
plt.ylabel('Kappa')
plt.show()

In [ ]:
nqe.plot_kappa_temperature(list_temperature, list_kappa)
nqe.plot_kappa_temperature_inv(list_temperature, list_kappa)

In [ ]:
rates = []
for temperature in list_temperature:
    output = nqe.run_ts(directory_ts,
                        atoms_ts,
                        driver=driver_code,
                        tol_energy=tol_energy,
                        tol_force=tol_force,
                        tol_position=tol_position,
                        total_steps=total_steps,
                        )

    atoms_ts, output_data_ts, _ = output

    rate = nqe.calc_forward_rate(n_atoms,
                                 directory_ts,
                                 directory_phonon_react,
                                 temperature,
                                 0.0)

    rates.append(rate)

In [ ]:
nqe.plot_arrhenius(list_temperature, rates)

rates_quantum = []
for i, rate in enumerate(rates):
    rates_quantum.append(rate * list_kappa[i])

nqe.plot_arrhenius_2(list_temperature, rates, rates_quantum)


In [ ]:
# Try and converge over beads
list_n_beads = [10, 20, 40, 60, 80]
list_kappa = []

for n_beads in list_n_beads:
    print(f"Running with {n_beads} beads")
    nqe.run_instanton(directory_instanton,
                      atoms_ts,
                      directory_ts,
                      driver=driver_code,
                      n_beads=n_beads,
                      temperature=temperature,
                      tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position,)
    kappa = nqe.calc_kappa_full(directory_ts,
                                directory_instanton,
                                temperature,
                                n_beads=n_beads,
                                react_energy=0.0,
                                )
    list_kappa.append(kappa)

In [ ]:
nqe.plot_bead_convergence(list_n_beads, list_kappa)

In [ ]:
# Classical KIE

# Run minimisation
output = nqe.run_optimise(directory_opti_d,
                          atoms,
                          driver=driver_code,
                          deuterate=True,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position,
                          )
atoms_opti_d, output_data_opti_d, _ = output

# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti_d, save=False)

nqe.run_phonons(directory_phonon_react_d,
                atoms_opti_d,
                driver=driver_code,
                deuterate=True
                )

eigvals = np.genfromtxt(os.path.join(directory_phonon_react_d, 'phonon.phonons.eigval'))
fr = nqe.freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

nqe.run_instanton_post_process(directory_phonon_react_d,
                               process_type='reactant',
                               temperature=temperature,
                               filter_list=n_atoms - 1)

output = nqe.run_ts(directory_ts_d,
                    atoms_ts,
                    driver=driver_code,
                    tol_energy=tol_energy,
                    tol_force=tol_force,
                    tol_position=tol_position,
                    total_steps=total_steps,
                    deuterate=True,
                    )

atoms_ts_d, output_data_ts_d, _ = output

rate = nqe.calc_forward_rate(n_atoms,
                             directory_ts_d,
                             directory_phonon_react_d,
                             temperature,
                             0.0)
print(rate)
